In [ ]:
import os

from langgraph.store.postgres import PostgresStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq


# ============================================================
# 1. SETUP
# ============================================================

DB_URI = "postgresql://postgres: your_password@localhost:5432/database_name"

os.environ["GROQ_API_KEY"] = "GROQ_API_KEY"


# Embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

embedding_dimension = len(
    embeddings.embed_query("Hello world")
)

print("Embedding dimensions:", embedding_dimension)


# Groq LLM
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)


# ============================================================
# 2. POSTGRES STORE
# ============================================================

with PostgresStore.from_conn_string(
    DB_URI,
    index={
        "dims": embedding_dimension,
        "embed": embeddings,
        "fields": ["text"],
    }
) as store:

    store.setup()

    user_id = "user_123"
    namespace = ("users", user_id, "memories")


    # ========================================================
    # 3. CREATE 10 MEMORIES
    # ========================================================

    memories = [
        "User is learning Python.",
        "User is learning LangGraph.",
        "User is learning LangChain.",
        "User is interested in RAG.",
        "User is interested in AI agents.",
        "User is learning tool calling.",
        "User wants to become an AI engineer.",
        "User is not interested in PostgreSQL.",
        "User prefers practical coding examples.",
        "User is building agentic AI projects.",
    ]

    for i, memory in enumerate(memories, 1):
        store.put(
            namespace,
            f"memory_{i}",
            {"text": memory}
        )

    print("\n10 memories saved.")


    # ========================================================
    # 4. USER QUERY
    # ========================================================

    user_query = "In which skill user is not interested"


    # ========================================================
    # 5. SEMANTIC SEARCH
    # ========================================================

    results = store.search(
        namespace,
        query=user_query,
        limit=1
    )


    print("\nUSER QUERY:")
    print(user_query)

    print("\nTOP 1 MATCHING MEMORIES:")

    for i, result in enumerate(results, 1):
        print(f"{i}. {result.value['text']}")


    # ========================================================
    # 6. SEND MEMORIES + QUERY TO LLM
    # ========================================================

    memory_context = "\n".join(
        f"- {result.value['text']}"
        for result in results
    )

    prompt = f"""
Relevant memories about the user:

{memory_context}

User question:

{user_query}

Answer the user's question using the relevant memories.
"""

    response = llm.invoke(prompt)

    print("\nAI ANSWER:")
    print(response.content)


print("\nPostgreSQL connection closed.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1704.55it/s]


Embedding dimensions: 384

10 memories saved.

USER QUERY:
In which skill user is not interested

TOP 1 MATCHING MEMORIES:
1. User is not interested in PostgreSQL.

AI ANSWER:
The user is not interested in the **PostgreSQL** skill.

PostgreSQL connection closed.
